# Lesson 02 — The clockwork solar system

**Phase 2 of the gravsim project: Newton, N bodies.**

Lesson 01 integrated two bodies. Now we add the other seven planets and let everything pull
on everything. The solar system of this lesson is the one Laplace had in mind when he called
it a clockwork: nine point masses, Newtonian gravity, nothing else. By the end you should be
able to

1. write the N-body equations of motion and explain why the Sun moves;
2. check a run with the conservation laws (energy, angular momentum, momentum);
3. show, in 3-D, the Sun's wobble about the barycentre and relate its size to Jupiter;
4. quantify how much each planet disturbs Mercury's orbit — the *planetary wobble* of its
   perihelion that will hide the GR signal in Lesson 04;
5. compare a Newtonian run with the JPL Horizons ephemeris and interpret the km-level drift.

## 1. The physics: N bodies

For $N$ point masses with gravitational parameters $GM_j$ at barycentric positions $\mathbf r_j$,

$$
\ddot{\mathbf r}_i \;=\; -\sum_{j\neq i} GM_j\,\frac{\mathbf r_i-\mathbf r_j}{|\mathbf r_i-\mathbf r_j|^3}.
$$

Only the *attracting* mass enters each term, so a body with $GM = 0$ (a test particle) feels
the field but exerts none. Three things are conserved exactly, and all three are used below as
checks on the numerics:

* total momentum $\sum_i GM_i\,\mathbf v_i$ — equivalently, the **barycentre** moves uniformly, and
  in the barycentric frame it stays at the origin;
* total angular momentum $\sum_i GM_i\,\mathbf r_i\times\mathbf v_i$;
* total energy $\tfrac12\sum_i GM_i v_i^2 - \sum_{i<j} GM_iGM_j/r_{ij}$.

(gravsim carries masses as $GM$, so the "energy" it returns is $G\times$ the physical energy.
Nothing changes in the conservation statements.)

**The Sun moves.** Jupiter's $GM$ is $1/1047$ of the Sun's at 5.2 AU, so the Sun orbits the
Sun–Jupiter barycentre at $5.2\,\mathrm{AU}/1048 \approx 0.005$ AU — about one solar radius —
with Jupiter's 11.9-year period. Forgetting this is the first item on the project's pitfalls
list: all measurements below are made *relative to the Sun* or *in the barycentric frame*,
never relative to a fixed origin.

**Bodies.** Sun plus the eight planets. Earth is represented by the Earth–Moon barycentre
(EMB) with the combined $GM$; the outer planets by their system barycentres with the system
$GM$s. Pluto, the Moon and the asteroids are left out; together they shift Mercury's
Newtonian perihelion rate by about $0.1''$/century.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

from gravsim import constants as C
from gravsim.analysis import (
    angular_momentum,
    energy,
    fit_secular_rate,
    perihelion_longitude_series,
    relative_energy_error,
    separation,
    state_to_elements,
    wrap_angle,
)
from gravsim.bodies import Trajectory
from gravsim.data.horizons import load_j2000, load_reference_series
from gravsim.forces import accel_newton
from gravsim.integrators import integrate, integrate_scipy
from gravsim.viz import BODY_COLORS, figure_grid, plot_orbits_3d, plot_separation

plt.rcParams["figure.dpi"] = 100

## 2. The data: the solar system at J2000.0

The initial state comes from JPL Horizons (cached in `gravsim/data/j2000.json`, no network
needed): barycentric positions and velocities in the J2000 *ecliptic* frame, $x$ toward the
equinox, $z$ toward the ecliptic pole. From each planet's state relative to the Sun we recover
its osculating Kepler elements.

In [ ]:
state = load_j2000()
r_rel, v_rel = state.relative_to("Sun")
print(f"{'body':8s} {'GM [AU^3/d^2]':>14s} {'a [AU]':>8s} {'e':>7s} {'i [deg]':>8s} {'P [yr]':>8s}")
for name in C.PLANETS:
    i = state.index(name)
    el = state_to_elements(r_rel[i], v_rel[i], C.GM_SUN + state.gm[i])
    print(f"{name:8s} {state.gm[i]:14.4e} {float(el['a']):8.3f} {float(el['e']):7.4f} "
          f"{math.degrees(float(el['inc'])):8.2f} {float(el['period']) / C.JULIAN_YEAR_DAYS:8.3f}")
r_b, v_b = state.barycenter()
print(f"\nbarycentre of the 9 bodies: |r| = {np.linalg.norm(r_b):.1e} AU, |v| = {np.linalg.norm(v_b):.1e} AU/d")
print(f"the Sun itself sits {np.linalg.norm(state.pos[0]) / C.R_SUN_AU:.2f} solar radii from the barycentre")

The barycentre is not *exactly* at the origin ($\sim10^{-6}$ AU) because Horizons' barycentre
includes the Moon, Pluto and the asteroids that we omit. It is small enough to ignore.

## 3. A first run: ten years

The force law is `accel_newton` (a numba-compiled pair loop). Following Lesson 01 we use the
production integrator, DOP853, sampled once a day, and keep a Yoshida4 run (half-day step,
176 steps per Mercury orbit) alongside it for the energy comparison of §5.

In [ ]:
years = 10.0
t_end = state.t + years * C.JULIAN_YEAR_DAYS
t_daily = state.t + np.arange(0.0, years * C.JULIAN_YEAR_DAYS + 1.0, 1.0)
traj = integrate_scipy(state, accel_newton, t_daily)
traj_y = integrate(state, accel_newton, t_end, 0.5, "yoshida4", save_every=2)
print(f"DOP853:   {traj.n_steps} samples, {traj.meta['nfev']} force evaluations, "
      f"max |ΔE/E| = {np.abs(relative_energy_error(traj)).max():.1e}")
print(f"Yoshida4: {traj_y.meta['n_steps']} steps of {traj_y.meta['dt']:.4f} d, "
      f"max |ΔE/E| = {np.abs(relative_energy_error(traj_y)).max():.1e}")

In [ ]:
fig, axes = figure_grid(2, projection="3d", panel=(5.5, 5.0))
plot_orbits_3d(traj, ax=axes[0])
axes[0].set_title(f"Sun + 8 planets, Newton, {years:.0f} yr")
plot_orbits_3d(traj, list(C.PLANETS[:4]), ax=axes[1])
axes[1].set_title("the inner planets")
for ax in axes:
    ax.view_init(elev=28, azim=-55)

The same picture, interactive (rotate, zoom, hover for the time stamp). This cell needs the
optional `plotly` package (`uv pip install plotly`); it is skipped gracefully otherwise.

In [ ]:
try:
    from gravsim.viz import orbits_3d_figure
    fig = orbits_3d_figure(traj, list(C.PLANETS[:4]))
    fig.update_layout(height=520)
    fig.show()
except ImportError as exc:
    print(exc)

## 4. The Sun's wobble

Over 60 years (five Jupiter orbits, two Saturn orbits) the Sun traces a loopy path of about
one solar radius around the barycentre. We draw it in units of $R_\odot$ and colour it by time.
Check the size against the estimate $a_{J}\,GM_J/(GM_\odot + GM_J)$.

In [ ]:
long = integrate(state, accel_newton, state.t + 60.0 * C.JULIAN_YEAR_DAYS, 2.0, "yoshida4", save_every=5)
sun = long.body("Sun")[0] / C.R_SUN_AU
t_yr = (long.t - long.t[0]) / C.JULIAN_YEAR_DAYS
fig = plt.figure(figsize=(6.5, 6.0), layout="constrained")
ax = fig.add_subplot(projection="3d")
sc = ax.scatter(sun[:, 0], sun[:, 1], sun[:, 2], c=t_yr, cmap="viridis", s=3)
ax.scatter([0], [0], [0], marker="+", color="k", s=80, label="barycentre")
lim = np.abs(sun).max() * 1.05
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_zlim(-lim, lim)
ax.set_box_aspect((1, 1, 1))
ax.set_xlabel("x [R_sun]"); ax.set_ylabel("y [R_sun]"); ax.set_zlabel("z [R_sun]")
ax.set_title("The Sun's path about the barycentre, 60 years (colour = time)")
fig.colorbar(sc, ax=ax, shrink=0.6, label="t [yr]")
ax.legend(loc="upper left")

a_J = float(state_to_elements(r_rel[state.index("Jupiter")], v_rel[state.index("Jupiter")], C.GM_SUN)["a"])
est = a_J * C.GM["Jupiter"] / (C.GM_SUN + C.GM["Jupiter"]) / C.R_SUN_AU
print(f"max |r_sun| = {np.linalg.norm(sun, axis=1).max():.2f} R_sun;  Jupiter alone would give {est:.2f} R_sun")

## 5. Conservation laws as a health check

Energy and angular momentum should be conserved to the integrator's accuracy; the barycentre
should not move at all (the pairwise force is exactly antisymmetric, so momentum is conserved
to roundoff by *every* integrator, even Euler). Note the two signatures from Lesson 01: the
symplectic Yoshida4 run has a *bounded* energy error of $2\times10^{-9}$ at a half-day step;
DOP853 at `rtol=1e-12` drifts, but only at the $10^{-13}$ level.

In [ ]:
for name, tr in (("DOP853", traj), ("Yoshida4 dt = 0.5 d", traj_y)):
    E = energy(tr.pos, tr.vel, tr.gm)
    L = angular_momentum(tr.pos, tr.vel, tr.gm)
    bary = np.einsum("tnk,n->tk", tr.pos, tr.gm) / tr.gm.sum()
    print(f"{name:20s} max |ΔE/E| = {np.abs(E / E[0] - 1).max():.1e}   "
          f"max |ΔL/L| = {np.abs(np.linalg.norm(L, axis=1) / np.linalg.norm(L[0]) - 1).max():.1e}   "
          f"max |Δ barycentre| = {np.linalg.norm(bary - bary[0], axis=1).max():.1e} AU")

## 6. Jupiter's pull on Mercury (and everybody else's)

Mercury's orbit is *not* a fixed ellipse once the other planets are present. Its osculating
elements breathe with the synodic periods of the perturbers, and its longitude of perihelion
$\varpi$ — the angle we will track to the arcsecond in the next lessons — drifts secularly at
about $531''$/century and wobbles on top of that.

First the plain separation between Mercury in a Sun–Mercury run and Mercury in the full run,
on the same time grid. The two start identical and diverge by many thousands of km within a
decade: the planetary perturbations are not subtle.

In [ ]:
two = load_j2000().subset(("Sun", "Mercury"))  # same barycentric state, other planets removed
traj2 = integrate_scipy(two, accel_newton, t_daily)
# compare Mercury *relative to the Sun* in both runs (the barycentres differ)
d = traj.relative("Mercury", "Sun")[0] - traj2.relative("Mercury", "Sun")[0]
fig, axes = figure_grid(1, panel=(7.0, 3.8))
plot_separation(traj.t, np.linalg.norm(d, axis=1), ax=axes[0], label="Mercury: 9-body minus 2-body")
axes[0].set_title("The other planets move Mercury by thousands of km per decade")

In [ ]:
# osculating elements of Mercury along the 9-body run
r_m, v_m = traj.relative("Mercury", "Sun")
gm_m = C.GM_SUN + C.GM["Mercury"]
el_t = state_to_elements(r_m, v_m, gm_m)
t_yr = (traj.t - traj.t[0]) / C.JULIAN_YEAR_DAYS
fig, axes = figure_grid(3, panel=(4.2, 3.4))
axes[0].plot(t_yr, el_t["a"], lw=0.8); axes[0].set_ylabel("a [AU]")
axes[1].plot(t_yr, el_t["e"], lw=0.8); axes[1].set_ylabel("e")
dvarpi = np.degrees(np.unwrap(el_t["varpi"]) - el_t["varpi"][0]) * 3600.0
axes[2].plot(t_yr, dvarpi, lw=0.8); axes[2].set_ylabel("Δϖ [arcsec]")
for ax in axes:
    ax.set_xlabel("t [yr]"); ax.grid(True, lw=0.3)
fig.suptitle("Mercury's osculating elements in the 9-body run: the planetary wobble")
fit = fit_secular_rate(traj.t, el_t["varpi"])
print(f"secular drift of ϖ over {years:.0f} yr: {fit.rate_arcsec_per_century:.1f} ± "
      f"{fit.rate_err_arcsec_per_century:.1f} arcsec/century (textbook, century average: 531.6)")

Which planet does what? Remove one planet at a time, rerun with the *same* integrator and step,
and fit the drift of $\varpi_{\rm full}-\varpi_{\rm without}$. We deliberately use the cheap
Yoshida4 here: its own spurious precession ($+2600''$/cy at this step, Lesson 01) is the same
in every run and cancels in the differences — the trick we will use on the GR signal. Ten
years is short compared with the long-period terms, so these are rough numbers; the
century-long values (Clemence 1947) are Venus 278, Earth 90, Mars 2.5, Jupiter 154,
Saturn 7.3, Uranus 0.14, Neptune 0.04 arcsec/century.

In [ ]:
t_full, w_full = perihelion_longitude_series(traj_y, "Mercury")
print(f"Yoshida4 run, dense fit: {fit_secular_rate(t_full, w_full).rate_arcsec_per_century:.0f} arcsec/cy "
      f"(planets ~569 + spurious ~2630: useless on its own)\n")
print(f"{'removed':8s} {'drift of Mercury ϖ due to it [arcsec/cy]':>42s}")
total = 0.0
for name in C.PLANETS[1:]:
    keep = tuple(n for n in C.DEFAULT_BODIES if n != name)
    tr = integrate(load_j2000().subset(keep), accel_newton, t_end, 0.5, "yoshida4", save_every=2)
    _, w = perihelion_longitude_series(tr, "Mercury")
    rate = fit_secular_rate(t_full, wrap_angle(w_full - w)).rate_arcsec_per_century
    total += rate
    print(f"{name:8s} {rate:42.1f}")
print(f"{'sum':8s} {total:42.1f}")

## 7. Against the real thing: JPL Horizons

gravsim also ships a 100-year Horizons series of the same nine bodies (`horizons_ref.npz`,
20-day samples). Horizons is based on the DE441 ephemeris, which integrates **General
Relativity** (the Einstein–Infeld–Hoffmann equations), the Moon, 343 asteroids, Pluto and the
solar oblateness. Our Newtonian nine-body run must therefore drift away from it; the question
is by how much. (DOP853 again, so that the drift is physics, not integrator error.)

In [ ]:
ref = load_reference_series()
ref10 = Trajectory(*(a[ref.t <= t_end + 1e-9] for a in (ref.t, ref.pos, ref.vel)), ref.gm, ref.names)
dop = integrate_scipy(state, accel_newton, ref10.t)  # samples exactly at the Horizons epochs

t_yr = (ref10.t - ref10.t[0]) / C.JULIAN_YEAR_DAYS
fig, axes = figure_grid(1, panel=(7.5, 4.2))
ax = axes[0]
print(f"{'body':8s} {'after 1 yr [km]':>16s} {'after 10 yr [km]':>17s}")
for name in C.PLANETS:
    sep_km = separation(dop, ref10, name) * C.AU_KM
    ax.plot(t_yr, sep_km, color=BODY_COLORS[name], lw=1.0, label=name)
    k1 = int(np.argmin(np.abs(t_yr - 1.0)))
    print(f"{name:8s} {sep_km[k1]:16.1f} {sep_km[-1]:17.1f}")
ax.set_yscale("log"); ax.set_xlabel("t [yr]"); ax.set_ylabel("|r_Newton - r_Horizons| [km]")
ax.set_title("Newtonian 9-body run (DOP853) versus the DE441-based Horizons ephemeris")
ax.legend(fontsize=8, ncol=2); ax.grid(True, lw=0.3, which="both")

Two things to notice.

* The drifts are kilometres to thousands of kilometres over a decade, against orbits of
  $10^8$ km: Newton with nine bodies is right to $10^{-5}$. The clockwork works.
* Mercury drifts the most in relative terms and *along its orbit*. That is the first sighting of
  the effect this course is about: Horizons contains GR, our run does not, and the difference
  accumulates as a slow turning of Mercury's ellipse. We will measure it in Lesson 03 and
  explain it in Lesson 04.

## 8. Summary

* Nine point masses and Newton's law reproduce the ephemeris to parts in $10^5$ over a decade.
* The Sun moves by about a solar radius; always work relative to the Sun or in the
  barycentric frame.
* Mercury's perihelion drifts at $\sim530''$/century because of the planets (Venus and Jupiter
  first), with a large periodic wobble on top.
* The residual against Horizons is dominated by GR for Mercury — the subject of Lessons 03–04.

## Exercises

1. Add a test particle (`gm = 0`) on a circular orbit at 2.8 AU (the asteroid belt) with
   `SystemState.with_` and integrate ten years. How large is its eccentricity after ten years?
   (Hint: `state_to_elements` along the trajectory.)
2. Repeat the "which planet" table with Yoshida4 at $\Delta t = 0.25$ d. The *absolute* drift of
   $\varpi$ in the full run changes by a factor 16 (why?), the *differences* barely do.
3. ★ The Sun's wobble has the periods of Jupiter and Saturn mixed. Fourier-transform
   $x_\odot(t)$ from a 200-year run and identify the two peaks.
4. ★ Horizons' Earth is the Earth–Moon barycentre too, but Horizons *integrates* the Moon
   separately. Estimate the size of the error we make by treating the EMB as a point mass
   (compare the Moon's quadrupole with the Sun's tidal field at 1 AU).